In [168]:
from pathlib import Path
import pandas as pd
import numpy as np

# Load the original dataset
DATA_PATH = Path("../data/processed/meajor_cleaned.csv")

df = pd.read_csv(DATA_PATH)

# Display dataset dimensions
print("Dataset shape:", df.shape)

# Display column names
print("\nColumns:")
print(df.columns.tolist())

# Display the first five rows
display(df.head())

Dataset shape: (108685, 20)

Columns:
['sender', 'sender_domain', 'receiver', 'receiver_domain', 'date', 'subject', 'content_types', 'body', 'urls', 'url_count', 'url_length_max', 'url_length_avg', 'url_subdom_max', 'url_subdom_avg', 'attachment_count', 'has_attachments', 'attachment_types', 'language', 'source', 'label']


,sender,sender_domain,receiver,receiver_domain,date,subject,content_types,body,urls,url_count,url_length_max,url_length_avg,url_subdom_max,url_subdom_avg,attachment_count,has_attachments,attachment_types,language,source,label
0,d66e9e64b006d6bca649f1c945129c42c43836872b2ead...,enron.com,35c5a9fb9fba3b8737ed7cef2a87e427a73db4fca85f6b...,enron.com,2001-06-29 09:37:04-05:00,[ORGANIZATION] failover plan.,text/plain,"Hi [NAME], \n\nTonight we are rolling out a n...",NaN,0.0,0.0,0.0,0.0,0.0,0.0,False,NaN,en,trec5,0.0
1,0907d5c64598aa2639154ed4e1556be615669e40052a1f...,enron.com,aa2c35499eae5999bf6080453cc719a891da2bb0c3803d...,enron.com,2001-06-29 08:39:30-05:00,RE: Intranet Site,text/plain,"[NAME] r these new?\tIntranet Site\n\n[NAME],\...",http://eastpower.dev.corp.enron.com/summary/pj...,3.0,60.0,58.0,3.0,3.0,0.0,False,NaN,en,trec5,0.0
2,7c3201a5ff8c5985218f1e3f11e330dc0242bbd28c6c20...,enron.com,a736837579feb601fbf6c0657d3d93689774afa6491bb9...,enron.com;enron.com,2001-06-29 10:35:17-05:00,FW: [ORGANIZATION] Company information,text/plain,"[NAME]/[NAME],\n\nWe are currently trading und...",NaN,0.0,0.0,0.0,0.0,0.0,0.0,False,NaN,en,trec5,0.0
3,8531d54a169c4af106b9ea2165d4986b8cc10fc0a6bb9b...,enron.com,765a3ec4a67e40118d22de5729b05d090a1b59cb443bf6...,enron.com;enron.com,2001-06-29 10:40:02-05:00,New Master Physical,text/plain,[NAME] and [NAME] -\n\nAttached is a worksheet...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,False,NaN,en,trec5,0.0
4,7c3201a5ff8c5985218f1e3f11e330dc0242bbd28c6c20...,enron.com,ce418c97ac415706338972e1dbbd99ebb8c617b5c937a3...,enron.com;enron.com;enron.com,2001-06-29 10:48:00-05:00,FW: [ORGANIZATION]/Mirant GISB,text/plain,FYI. Below is a copy of my communication with ...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,False,NaN,en,trec5,0.0


### 🧹 Step 4: Investigate Extreme URL Features

Our EDA identified a maximum URL length of 66,647 characters. That's an extreme value, but we need to investigate it before deciding what to do with it.

In [169]:
# Identify columns related to URLs
url_columns = [
    col for col in df.columns
    if "url" in col.lower()
]

print("URL-related columns:")
print(url_columns)

URL-related columns:
['urls', 'url_count', 'url_length_max', 'url_length_avg', 'url_subdom_max', 'url_subdom_avg']


In [170]:
url_features = [
    'urls',
    'url_count',
    'url_length_max',
    'url_length_avg',
    'url_subdom_max',
    'url_subdom_avg'
]

display(df[url_features].describe().T)

,count,mean,std,min,25%,50%,75%,max
url_count,108685.0,3.308138,10.507161,0.0,0.0,1.000000,2.0,553.0
url_length_max,108685.0,31.371091,205.156377,0.0,0.0,26.000000,48.0,66647.0
url_length_avg,108685.0,26.691328,203.959800,0.0,0.0,24.500000,43.5,66647.0
url_subdom_max,108685.0,0.659539,0.791752,0.0,0.0,1.000000,1.0,5.0
url_subdom_avg,108685.0,0.585495,0.712159,0.0,0.0,0.208333,1.0,5.0


In [171]:
# Inspect the five records with the longest URLs
# Identify records with the largest maximum URL lengths
extreme_urls = df.nlargest(5, 'url_length_max')

display(
    extreme_urls[
        [
            'urls',
            'url_count',
            'url_length_max',
            'url_length_avg',
            'url_subdom_max',
            'url_subdom_avg'
        ]
    ]
)

,urls,url_count,url_length_max,url_length_avg,url_subdom_max,url_subdom_avg
74569,http://www.zief.pl/iraq.jpgR0lGODdhaAFHAfYAAIE...,1.0,66647.0,66647.000000,1.0,1.000000
24342,http://alerts.yahoo.com\nhttp://finance.yahoo....,17.0,629.0,77.705882,2.0,1.470588
24343,http://alerts.yahoo.com\nhttp://finance.yahoo....,16.0,629.0,78.562500,2.0,1.437500
68785,http://us.js2.yimg.com/us.js.yimg.com/lib/pim/...,24.0,612.0,135.708333,2.0,1.208333
89308,http://pics.ebaystatic.com/aw/pics/logos/ebay_...,47.0,471.0,60.063830,1.0,0.936170


In [172]:
print("Data type:", df['urls'].dtype)

display(df['urls'].head(10))

print("\nMissing values:", df['urls'].isna().sum())

Data type: str


0                                                  NaN
1    http://eastpower.dev.corp.enron.com/summary/pj...
2                                                  NaN
3                                                  NaN
4                                                  NaN
5                                                  NaN
6                                                  NaN
7                                                  NaN
8                                                  NaN
9                                                  NaN
Name: urls, dtype: str


Missing values: 44633


In [173]:
# Inspect non-missing URL values
non_missing_urls = df.loc[
    df['urls'].notna(),
    'urls'
]

for idx, value in non_missing_urls.head(5).items():
    print(f"\nRow index: {idx}")
    print(f"Stored value length: {len(value)}")
    print(f"Preview: {value[:300]}")


Row index: 1
Stored value length: 176
Preview: http://eastpower.dev.corp.enron.com/summary/pjmsummary.asp
http://eastpower.dev.corp.enron.com/summary/nyisosummary.asp
http://eastpower.dev.corp.enron.com/Cooper/MID_NYISO.ASP

Row index: 15
Stored value length: 76
Preview: http://home.enron.com/sites/
http://community.zuniversity.com/z/EnronCougars

Row index: 16
Stored value length: 60
Preview: http://www.patriotichouse.com
http://www.patriotichouse.com"

Row index: 21
Stored value length: 147
Preview: http://www20.cera.com/eprofile?u=35&m=2527
http://www20.cera.com/client/updateaccount
http://www20.cera.com/client/forgot
http://www20.cera.com/tos

Row index: 34
Stored value length: 23
Preview: http://www.williams.com


In [174]:
# Records where the URL field is missing
missing_url_mask = df['urls'].isna()

print(
    "Missing URL values:",
    missing_url_mask.sum()
)

print(
    "\nMissing URL values with positive URL counts:",
    (
        missing_url_mask
        & df['url_count'].gt(0)
    ).sum()
)

print(
    "Missing URL values with zero URL counts:",
    (
        missing_url_mask
        & df['url_count'].eq(0)
    ).sum()
)

print(
    "Missing URL values with missing URL counts:",
    (
        missing_url_mask
        & df['url_count'].isna()
    ).sum()
)

Missing URL values: 44633

Missing URL values with positive URL counts: 0
Missing URL values with zero URL counts: 44633
Missing URL values with missing URL counts: 0


In [175]:
# Inspect the suspicious record
# Inspect the suspicious record
row_idx = 74569

columns_to_inspect = [
    'sender_domain',
    'date',
    'subject',
    'content_types',
    'body',
    'urls',
    'url_count',
    'url_length_max',
    'url_length_avg',
    'attachment_count',
    'has_attachments',
    'attachment_types',
    'source',
    'label'
]

display(df.loc[[row_idx], columns_to_inspect].T)

,74569
sender_domain,za3.so-net.ne.jp
date,2007-04-25 16:58:31+09:00
subject,[ORGANIZATION] Blog prediction: [ORGANIZATION]...
content_types,text/plain;text/html
body,I read that the first thing the brothers will ...
urls,http://www.zief.pl/iraq.jpgR0lGODdhaAFHAfYAAIE...
url_count,1.0
url_length_max,66647.0
url_length_avg,66647.0
attachment_count,0.0


In [176]:
# Extract the suspicious URL field
suspicious_urls = df.loc[row_idx, 'urls']

print("Stored value length:", len(suspicious_urls))

print("\nFirst 500 characters:")
print(suspicious_urls[:500])

print("\nLast 300 characters:")
print(suspicious_urls[-300:])

print("\nNumber of newline-separated entries:")
print(len(suspicious_urls.splitlines()))

Stored value length: 66647

First 500 characters:
http://www.zief.pl/iraq.jpgR0lGODdhaAFHAfYAAIEWz10IY/y0yEYUK0KRdlNJahaSlbfANztCTnx5I1CGgcfZy8imHotIbS9mNVl3c9fg3DzPFFY/kJptHplHRz9Abzpxxzt8A1AjEnp5US54OJ4+0oV5REyQhYDLAZWWnylpRGxwXVVyGKpISejuuhhiN9TPpzLfQWHMbGxqnKM+a0txS41rgEF+iDopWH6KJEMITn10uIVkadHx1FEzU1bXTT44FFepgn/VJFBRXod/UZt2pFq9Jh2NTcHF9Ny9sU01djZTBI6vsYOSC0BK0GjOWm1SRwtkiSe8x6V0Nrm1gYmnGTLWHSnigrP4yKsqs7b35W06QofLUkvEgRNwq0toY0ooInU6CGtXCVIuyma7MlkxbGGcw1ZHKi9NlxyfxgmKQDvkcmyxvj6WY2UClaU9eSZScIJhZWpBXnWnYYVkhUWL17VxaLy5D2qEXE/PQSeFso9Bb

Last 300 characters:
/j4tft/zVA/GqXkPrn+Mqd4Z+amlVFuwYyQ6Xsew3d0OTN22nZizJC8VZjR/GYjbDX31Xd3EIeTlWmEF7Gpm2eMPjKFAvixaPmCmJpDlR553OD7fSz7mYMc0+QflGGXkvUXlYN5MtO3mF6eFa5V4WqPjbeY9YA7oFpPkVv7j673n+TetXG5UkMbjJe7byh0wTv5JtfVZV2nn1uLjYq7nZa5Jft7lz/Dlmx7mmjXmWe7km6pYaL6bav7gG57Gen5xLjurUU7nkUd4pY7n1O3pqm7mlPfnhVboVc4IgQAAOw==

Number of newline-separated entries:
1


In [177]:
# Check records with zero URLs but nonzero URL lengths
zero_url_inconsistent = df[
    (df['url_count'] == 0)
    & (
        (df['url_length_max'] != 0)
        | (df['url_length_avg'] != 0)
        | (df['url_subdom_max'] != 0)
        | (df['url_subdom_avg'] != 0)
    )
]

print(
    "Records with zero URLs but nonzero URL features:",
    len(zero_url_inconsistent)
)

display(zero_url_inconsistent.head())

Records with zero URLs but nonzero URL features: 0


,sender,sender_domain,receiver,receiver_domain,date,subject,content_types,body,urls,url_count,url_length_max,url_length_avg,url_subdom_max,url_subdom_avg,attachment_count,has_attachments,attachment_types,language,source,label


In [178]:
# An average should not exceed the maximum
invalid_length_relationship = df[
    df['url_length_avg'] > df['url_length_max']
]

print(
    "Records where average URL length exceeds maximum:",
    len(invalid_length_relationship)
)

display(
    invalid_length_relationship[
        [
            'urls',
            'url_count',
            'url_length_max',
            'url_length_avg'
        ]
    ].head(10)
)

Records where average URL length exceeds maximum: 0


,urls,url_count,url_length_max,url_length_avg


In [179]:
# Compare attachment count with attachment indicator
attachment_mismatches = df[
    (
        df['attachment_count'] > 0
    ) != (
        df['has_attachments'] == True
    )
]

print(
    "Attachment count/indicator mismatches:",
    len(attachment_mismatches)
)

display(
    attachment_mismatches[
        [
            'attachment_count',
            'has_attachments',
            'attachment_types'
        ]
    ].head(10)
)

Attachment count/indicator mismatches: 0


,attachment_count,has_attachments,attachment_types


In [180]:
suspicious_records = df[
    df["url_length_max"] == 66647
]

print("Matching records:", len(suspicious_records))

Matching records: 1


In [181]:
url_numeric_features = [
    "url_count",
    "url_length_max",
    "url_length_avg",
    "url_subdom_max",
    "url_subdom_avg"
]

for col in url_numeric_features:
    print(f"\n{col}")
    print("Missing values:", df[col].isna().sum())
    print("Negative values:", (df[col] < 0).sum())


url_count
Missing values: 0
Negative values: 0

url_length_max
Missing values: 0
Negative values: 0

url_length_avg
Missing values: 0
Negative values: 0

url_subdom_max
Missing values: 0
Negative values: 0

url_subdom_avg
Missing values: 0
Negative values: 0


In [182]:
df.isnull().sum()

sender                 494
sender_domain          494
receiver              2371
receiver_domain       2371
date                     0
subject               1455
content_types            1
body                     1
urls                 44633
url_count                0
url_length_max           0
url_length_avg           0
url_subdom_max           0
url_subdom_avg           0
attachment_count         0
has_attachments          0
attachment_types    107019
language                 1
source                   1
label                    1
dtype: int64

### Step 1: Handle missing values in urls

Your urls column has 44,633 missing values. We want the model to distinguish between an email that contains no URL and one that contains a URL.

Replace the missing values with an empty string or a placeholder. For this project, I recommend an empty string ("") because it represents the absence of a URL without introducing artificial text.

In [183]:
# Handle both NaN values and empty strings
df["urls"] = (
    df["urls"]
    .fillna("[NO URL]")
    .replace(r"^\s*$", "[NO URL]", regex=True)
)

# Verify
print("Empty strings:", df["urls"].eq("").sum())
print("Missing values:", df["urls"].isna().sum())
print("NO URL placeholders:", df["urls"].eq("[NO URL]").sum())

print("\nURL examples:")
print(df["urls"].head(10).tolist())

Empty strings: 0
Missing values: 0
NO URL placeholders: 44633

URL examples:
['[NO URL]', 'http://eastpower.dev.corp.enron.com/summary/pjmsummary.asp\nhttp://eastpower.dev.corp.enron.com/summary/nyisosummary.asp\nhttp://eastpower.dev.corp.enron.com/Cooper/MID_NYISO.ASP', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]']


In [184]:
df["has_url"] = df["url_count"].gt(0)

print(df["has_url"].value_counts(dropna=False))

has_url
True     64052
False    44633
Name: count, dtype: int64


In [185]:
print(
    df.loc[
        df["has_url"] == False,
        "urls"
    ].head(50).tolist()
)

['[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]', '[NO URL]']


In [186]:
# Calculate missing-value statistics
missing_stats = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2)
})

# Display only columns with missing values
missing_stats = (
    missing_stats[missing_stats["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

display(missing_stats)

,missing_count,missing_percentage
attachment_types,107019,98.47
receiver_domain,2371,2.18
receiver,2371,2.18
subject,1455,1.34
sender,494,0.45
sender_domain,494,0.45
content_types,1,0.00
body,1,0.00
language,1,0.00
source,1,0.00


In [187]:
# Investigate the sender and receiver information
# Missing sender information
missing_sender = df[
    df["sender"].isnull() |
    df["sender_domain"].isnull()
]

print("Rows with missing sender information:", len(missing_sender))

display(
    missing_sender[["sender", "sender_domain"]].head(10)
)

# Missing receiver information
missing_receiver = df[
    df["receiver"].isnull() |
    df["receiver_domain"].isnull()
]

print("Rows with missing receiver information:", len(missing_receiver))

display(
    missing_receiver[["receiver", "receiver_domain"]].head(10)
)

Rows with missing sender information: 494


,sender,sender_domain
287,NaN,NaN
340,NaN,NaN
386,NaN,NaN
445,NaN,NaN
450,NaN,NaN
521,NaN,NaN
539,NaN,NaN
648,NaN,NaN
651,NaN,NaN
658,NaN,NaN


Rows with missing receiver information: 2371


,receiver,receiver_domain
162,NaN,NaN
209,NaN,NaN
249,NaN,NaN
303,NaN,NaN
344,NaN,NaN
368,NaN,NaN
403,NaN,NaN
445,NaN,NaN
446,NaN,NaN
449,NaN,NaN


In [188]:
# Investigate the  missing subjects
# Examine emails with missing subjects
missing_subject = df[df["subject"].isnull()]

print("Rows with missing subject:", len(missing_subject))

display(
    missing_subject[["sender", "receiver", "subject", "body", "label"]]
    .head(10)
)

Rows with missing subject: 1455


,sender,receiver,subject,body,label
11,4fe8253faf675c7f6155f2dffed368349669f7889ab3d8...,6ec4ed45ee48057e093ebba1154ace85fa4ba72239f670...,NaN,Per [NAME] -\n\nAttached you will find the Str...,0.0
22,f0f5ffc8181fcdf5c9106ce360d40030522accc981be88...,5814d807b8a462faa4f8dd9870341ff20abdff30804c3b...,NaN,"[USERNAME] \n here are the files, just in cas...",0.0
88,b223d3d66d58fef340544ed41263b5b9fcbe0413954754...,6faa355581a5143ffd7653a4e4e4f40854d653f3596ba0...,NaN,Enquire within for\nBrand new [PRODUCT] s0ftab...,1.0
127,6c2836d0bfb1d239ea7e5624224b53bb4b950690072cf7...,5a37ed79e09fa97afff4b9f6e3126e907a3cf6cddae9dd...,NaN,Want a Watch?\n[URL],1.0
191,b6700563535dcdfb12ee7c8670b8d2c34226f6c27851cb...,765a3ec4a67e40118d22de5729b05d090a1b59cb443bf6...,NaN,Hi.\n\n Below is the tpa initiated by [ORG...,0.0
277,87450134aa305e2f76e313f59469647a9d8020ce6bdc2c...,bc06a1664865f2574255db49eddcffe1493fbe784cbfc0...,NaN,"Deal number for desk-desk, [REFERENCE_NUMBER],...",0.0
340,NaN,76003082687cb41c892dc31afc3c46e75c72d77addf272...,NaN,is speech giving field \nlived expected confid...,1.0
356,5d6bde625bc1bed6fe7e6654644810b520848a7335f4df...,70ee1ad87833efd400aee6afaae7fb7aa877054bdfb74c...,NaN,Want a cheap Watch?\n[URL],1.0
445,NaN,NaN,NaN,""" <|EMAIL|>\nTo: <|EMAIL|>\nMessage-Id: <|EMAI...",1.0
450,NaN,980b2dac42b4e9e3f6c7115ec25f5a0129a856d562494c...,NaN,parallel company understand stayed \nlanguage ...,1.0


In [189]:
# Inspect rows with missing content metadata or body
missing_content = df[
    df["content_types"].isnull() |
    df["body"].isnull()
]

display(missing_content)

,sender,sender_domain,receiver,receiver_domain,date,subject,content_types,body,urls,url_count,...,url_length_avg,url_subdom_max,url_subdom_avg,attachment_count,has_attachments,attachment_types,language,source,label,has_url
9,0ac8edf9f46533ba774c9834c14614fc04663645614b76...,enron.com,1aef9ef79ea552c8908beaf58a9422f7f6c18995363b80...,enron.com,2001-06-29 15:16:54-05:00,"Accepted: Internet Meeting - [NAME], [NAME], [...",NaN,NaN,[NO URL],0.0,...,0.0,0.0,0.0,0.0,False,NaN,unk,trec5,0.0,False


In [190]:
# Examine missing attachment types
missing_attachment_types = df[df["attachment_types"].isnull()]

print("Rows with missing attachment types:", len(missing_attachment_types))

# Compare against attachment indicators
display(
    missing_attachment_types[
        ["has_attachments", "attachment_count", "attachment_types"]
    ].head(20)
)

Rows with missing attachment types: 107019


,has_attachments,attachment_count,attachment_types
0,False,0.0,NaN
1,False,0.0,NaN
2,False,0.0,NaN
3,False,0.0,NaN
4,False,0.0,NaN
5,False,0.0,NaN
6,False,0.0,NaN
7,False,0.0,NaN
8,False,0.0,NaN
9,False,0.0,NaN


In [191]:
# Inspect missing language and source values
missing_metadata = df[
    df["language"].isnull() |
    df["source"].isnull()
]

display(missing_metadata)

,sender,sender_domain,receiver,receiver_domain,date,subject,content_types,body,urls,url_count,...,url_length_avg,url_subdom_max,url_subdom_avg,attachment_count,has_attachments,attachment_types,language,source,label,has_url
51258,57b11276b707f18ed9c94f8e7cfe4106062bb24480ff63...,advalvas.be,247e0a5daf10ac2f4a2ec564931b09941a066488d3fa53...,media.mit.edu,1994-09-02 05:19:11+04:00,(<|SIMBOL|>Ü 5<|SIMBOL|>Ð<|SIMBOL|><|SIMBOL|> ...,text/plain;text/html,dangerous cork blob thirteenth alliterate argu...,[NO URL],0.0,...,0.0,0.0,0.0,0.0,False,NaN,NaN,NaN,NaN,False


In [192]:
# Compare missing attachment types with attachment indicators

attachment_summary = (
    df.assign(
        attachment_type_missing=df["attachment_types"].isna()
    )
    .groupby(
        ["attachment_type_missing", "has_attachments"],
        dropna=False
    )
    .size()
    .unstack(fill_value=0)
)

display(attachment_summary)

has_attachments,False,True
attachment_type_missing,,
False,0,1666
True,107019,0


In [193]:
# Represent emails without attachments explicitly
df["attachment_types"] = df["attachment_types"].fillna("none")

print("Missing attachment types:", df["attachment_types"].isna().sum())
print(df["attachment_types"].value_counts().head(10))

Missing attachment types: 0
attachment_types
none                                     107019
application/msword                          383
application/octet-stream                    309
text/plain                                  111
application/vnd.ms-excel                    106
text/x-vcard                                 93
application/pgp-signature                    66
application/msword;application/msword        55
application/pdf                              46
image/jpeg                                   34
Name: count, dtype: int64


In [194]:
# Emails that have attachments but lack attachment type information

missing_types_with_attachments = df[
    df["attachment_types"].isna()
    & (
        df["has_attachments"].eq(True)
        | df["attachment_count"].gt(0)
    )
]

print(
    "Records with attachments but missing attachment types:",
    len(missing_types_with_attachments)
)

display(
    missing_types_with_attachments[
        [
            "has_attachments",
            "attachment_count",
            "attachment_types"
        ]
    ].head(10)
)

Records with attachments but missing attachment types: 0


,has_attachments,attachment_count,attachment_types


In [195]:
# Check whether sender and sender_domain are missing together

sender_check = pd.crosstab(
    df["sender"].isna(),
    df["sender_domain"].isna()
)

display(sender_check)

# Check whether receiver and receiver_domain are missing together

receiver_check = pd.crosstab(
    df["receiver"].isna(),
    df["receiver_domain"].isna()
)

display(receiver_check)

sender_domain,False,True
sender,,
False,108191,0
True,0,494


receiver_domain,False,True
receiver,,
False,106314,0
True,0,2371


In [196]:
# Fill missing sender and receiver metadata
metadata_columns = [
    "sender",
    "sender_domain",
    "receiver",
    "receiver_domain"
]

df[metadata_columns] = df[metadata_columns].fillna("unknown")

print(df[metadata_columns].isnull().sum())

sender             0
sender_domain      0
receiver           0
receiver_domain    0
dtype: int64


In [197]:
missing_content = df[
    df["content_types"].isna()
    | df["body"].isna()
]

display(missing_content.T)

,9
sender,0ac8edf9f46533ba774c9834c14614fc04663645614b76...
sender_domain,enron.com
receiver,1aef9ef79ea552c8908beaf58a9422f7f6c18995363b80...
receiver_domain,enron.com
date,2001-06-29 15:16:54-05:00
subject,"Accepted: Internet Meeting - [NAME], [NAME], [..."
content_types,NaN
body,NaN
urls,[NO URL]
url_count,0.0


In [198]:
df["body"] = (
    df["body"]
    .fillna("[NO BODY]")
    .replace(r"^\s*$", "[NO BODY]", regex=True)
)
df["content_types"] = df["content_types"].fillna("unknown")

In [199]:
# Replace missing subjects with empty strings
df["subject"] = (
    df["subject"]
    .fillna("[NO SUBJECT]")
    .replace(r"^\s*$", "[NO SUBJECT]", regex=True)
)

print("Missing subjects:", df["subject"].isna().sum())

Missing subjects: 0


In [200]:
# Inspect the record with a missing target label
missing_label = df[df["label"].isnull()]

display(missing_label.T)

,51258
sender,57b11276b707f18ed9c94f8e7cfe4106062bb24480ff63...
sender_domain,advalvas.be
receiver,247e0a5daf10ac2f4a2ec564931b09941a066488d3fa53...
receiver_domain,media.mit.edu
date,1994-09-02 05:19:11+04:00
subject,(<|SIMBOL|>Ü 5<|SIMBOL|>Ð<|SIMBOL|><|SIMBOL|> ...
content_types,text/plain;text/html
body,dangerous cork blob thirteenth alliterate argu...
urls,[NO URL]
url_count,0.0


In [201]:
# Remove the record with a missing target label
df = df.dropna(subset=["label"]).copy()

print("Missing labels:", df["label"].isna().sum())
print("Dataset shape:", df.shape)

Missing labels: 0
Dataset shape: (108684, 21)


In [202]:
df.isnull().sum()

sender              0
sender_domain       0
receiver            0
receiver_domain     0
date                0
subject             0
content_types       0
body                0
urls                0
url_count           0
url_length_max      0
url_length_avg      0
url_subdom_max      0
url_subdom_avg      0
attachment_count    0
has_attachments     0
attachment_types    0
language            0
source              0
label               0
has_url             0
dtype: int64

In [203]:
CLEANED_DATA_PATH = Path("../data/processed/email_cleaned.csv")

CLEANED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(CLEANED_DATA_PATH, index=False)

print(f"Dataset saved to: {CLEANED_DATA_PATH}")

Dataset saved to: ..\data\processed\email_cleaned.csv
